# Meaningful Information in Retokenized Text — All Plots

Combined notebook for generating all plots from both wikitext and code experiments.
Data lives in `wikitext/` and `code/` subdirectories.

In [ ]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

WIKI = Path("wikitext")
CODE = Path("code")

def load_json(directory, name):
    path = directory / f"{name}.json"
    if not path.exists():
        print(f"Not found: {path}")
        return None
    return json.loads(path.read_text())

def get_slopes(data, condition="retok"):
    slopes_a, slopes_b = [], []
    for rec in data["records"]:
        slopes_b.append(rec["slope_b"])
        if condition in rec["conditions"]:
            for rt in rec["conditions"][condition]:
                v = rt["slope_a"]
                if math.isfinite(v):
                    slopes_a.append(v)
    return slopes_a, slopes_b

def get_slopes_simple(data):
    """For batch_p1p0.json format (no 'conditions' key)."""
    slopes_a, slopes_b = [], []
    for rec in data["records"]:
        slopes_b.append(rec["slope_b"])
        for rt in rec["retokenizations"]:
            slopes_a.append(rt["slope_a"])
    return slopes_a, slopes_b

---
# Part I — Wikitext (natural language)

## 1. Single-example cumulative −logprob

In [ ]:
d = load_json(WIKI, "example")

neg_lp_a = -np.array(d["logprobs_a"])
neg_lp_b = -np.array(d["logprobs_b"])
cum_a = np.cumsum(neg_lp_a)
cum_b = np.cumsum(neg_lp_b)
T = len(neg_lp_a)
positions = np.arange(1, T + 1)

slope_a = cum_a[-1] / T
slope_b = cum_b[-1] / T

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax = axes[0]
ax.plot(positions, cum_a, "-", color="crimson", lw=1.5,
        label=f"with retok prefix (p=1.0, {d['L_retok']} tokens)")
ax.plot(positions, cum_b, "-", color="steelblue", lw=1.5,
        label="canon only (no prefix)")
ax.plot([1, T], [slope_a, slope_a * T], ":", color="crimson", alpha=0.5, lw=1)
ax.plot([1, T], [slope_b, slope_b * T], ":", color="steelblue", alpha=0.5, lw=1)
ax.text(T * 0.6, cum_a[-1] * 0.7, f"slope \u2248 {slope_a:.2f} nats/tok",
        fontsize=8, color="crimson")
ax.text(T * 0.3, cum_b[-1] * 0.7, f"slope \u2248 {slope_b:.2f} nats/tok",
        fontsize=8, color="steelblue")
ax.set_ylabel("cumulative \u2212log P (nats)")
ax.set_title("Cumulative negative log-probability (slope \u2248 entropy rate)")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
diff = cum_b - cum_a
ax.plot(positions, diff, "-", color="darkgreen", lw=1.5)
ax.axhline(0, color="black", lw=0.5, ls="--")
ax.set_xlabel("token position t")
ax.set_ylabel("cumulative \u2212logP difference (B \u2212 A)")
ax.set_title("Information gained from retokenized prefix")
ax.grid(True, alpha=0.3)

fig.suptitle(f"OLMo-2-7B \u2014 cumulative \u2212logprob with/without retokenized prefix\n"
             f"Single wikitext passage, {T} tokens, p=1.0", y=1.01)
fig.tight_layout()
plt.show()

print(f"Entropy rate: A={slope_a:.2f}  B={slope_b:.2f} nats/token")

## 2. Tokenization segmentation visualization

In [ ]:
d = load_json(WIKI, "example")
canon_tokens = d["canon_tokens"]
retok_tokens = d["retok_tokens"]
text = d["text"]

colors_a = ["#aec7e8", "#c7b8ea"]
colors_b = ["#ffbb78", "#98df8a"]

total_chars = len(text)
row_width = 80
n_rows = (total_chars + row_width - 1) // row_width

fig_height = max(4, n_rows * 2.2 + 1.5)
fig, ax = plt.subplots(figsize=(16, fig_height))

y_offset = n_rows * 2.2

for row in range(n_rows):
    char_start = row * row_width
    char_end = min((row + 1) * row_width, total_chars)

    def get_row_tokens(tokens, char_start, char_end):
        pos = 0
        row_toks = []
        for tok in tokens:
            tok_start = pos
            tok_end = pos + len(tok)
            if tok_end > char_start and tok_start < char_end:
                display_tok = tok[max(0, char_start - tok_start):len(tok) - max(0, tok_end - char_end)]
                clip_start = max(tok_start, char_start) - char_start
                row_toks.append((clip_start, display_tok, tok))
            pos = tok_end
        return row_toks

    canon_row = get_row_tokens(canon_tokens, char_start, char_end)
    retok_row = get_row_tokens(retok_tokens, char_start, char_end)

    y_canon = y_offset - row * 2.2
    y_retok = y_canon - 0.9

    for x_pos, display_tok, full_tok in canon_row:
        width = len(display_tok)
        idx = canon_row.index((x_pos, display_tok, full_tok))
        color = colors_a[idx % 2]
        rect = mpatches.FancyBboxPatch(
            (x_pos, y_canon), width, 0.65, boxstyle="round,pad=0.03",
            facecolor=color, edgecolor="white", linewidth=0.5, alpha=0.85)
        ax.add_patch(rect)
        if width >= 1:
            fs = min(5.5, max(3, width * 1.5))
            ax.text(x_pos + width / 2, y_canon + 0.325,
                    display_tok.replace(" ", "\u00b7"),
                    ha="center", va="center", fontsize=fs, fontfamily="monospace")

    for x_pos, display_tok, full_tok in retok_row:
        width = len(display_tok)
        idx = retok_row.index((x_pos, display_tok, full_tok))
        color = colors_b[idx % 2]
        rect = mpatches.FancyBboxPatch(
            (x_pos, y_retok), width, 0.65, boxstyle="round,pad=0.03",
            facecolor=color, edgecolor="white", linewidth=0.5, alpha=0.85)
        ax.add_patch(rect)
        if width >= 1:
            fs = min(5.5, max(3, width * 1.5))
            ax.text(x_pos + width / 2, y_retok + 0.325,
                    display_tok.replace(" ", "\u00b7"),
                    ha="center", va="center", fontsize=fs, fontfamily="monospace")

    ax.text(-1.5, y_canon + 0.3, "E0", ha="right", va="center",
            fontsize=7, fontweight="bold", color="#1f77b4")
    ax.text(-1.5, y_retok + 0.3, "E'", ha="right", va="center",
            fontsize=7, fontweight="bold", color="#d62728")

ax.set_xlim(-4, row_width + 2)
ax.set_ylim(-0.5, y_offset + 1.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"Canonical (E0, {d['L_canon']} tokens) vs Retokenized (E', {d['L_retok']} tokens, p=1.0)\n"
             f"Same text, different segmentation \u2014 \u00b7 marks spaces",
             fontsize=10, pad=10)
fig.tight_layout()
plt.show()

## 3. Batch slopes — entropy rate by condition and p

In [ ]:
d = load_json(WIKI, "batch_p1p0")
records = d["records"]
p_values = d["config"]["retok_p_values"]
all_slopes_b = [rec["slope_b"] for rec in records]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

ax = axes[0, 0]
slopes_b_sorted = sorted([(rec["slope_b"], rec["passage_idx"]) for rec in records])
indices = [slopes_b_sorted[i][1] for i in [0, len(slopes_b_sorted)//4,
           len(slopes_b_sorted)//2, 3*len(slopes_b_sorted)//4, -1]]
colors = plt.get_cmap("tab10")
for k, idx in enumerate(indices):
    rec = records[idx]
    slopes_a_p1 = [rt["slope_a"] for rt in rec["retokenizations"] if rt["p"] == 1.0]
    ax.hist(slopes_a_p1, bins=15, alpha=0.5, color=colors(k),
            label=f"passage {idx} (slope_b={rec['slope_b']:.2f})", density=True)
    ax.axvline(rec["slope_b"], color=colors(k), ls="--", lw=1.5)
ax.set_xlabel("slope_a (entropy rate with retok prefix, nats/tok)")
ax.set_ylabel("density")
ax.set_title("A) Slope distribution for fixed passages (p=1.0)\ndashed = slope_b (no prefix)")
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for p in p_values:
    all_sa = [rt["slope_a"] for rec in records for rt in rec["retokenizations"] if rt["p"] == p]
    ax.hist(all_sa, bins=40, alpha=0.4, density=True, label=f"slope_a (p={p})")
ax.hist(all_slopes_b, bins=40, alpha=0.5, color="steelblue", density=True, label="slope_b (no prefix)")
ax.set_xlabel("entropy rate (nats/tok)"); ax.set_ylabel("density")
ax.set_title("B) Pooled slope distributions"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
cmap = plt.get_cmap("viridis")
for i, p in enumerate(p_values):
    xs, ys = [], []
    for rec in records:
        for rt in rec["retokenizations"]:
            if rt["p"] == p:
                xs.append(rec["slope_b"]); ys.append(rt["slope_a"])
    color = cmap(i / max(len(p_values) - 1, 1))
    ax.scatter(xs, ys, s=8, alpha=0.3, color=color, edgecolor="none", label=f"p={p}")
lims = [0, max(max(all_slopes_b) * 1.1, 3.5)]
ax.plot(lims, lims, "k--", lw=0.8, alpha=0.5, label="y = x")
ax.set_xlim(lims); ax.set_ylim([0, max(lims[1], 1.0)])
ax.set_xlabel("slope_b (no prefix, nats/tok)"); ax.set_ylabel("slope_a (with retok prefix, nats/tok)")
ax.set_title("C) Entropy rate: with vs without retok prefix"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
data_by_p = [[rt["slope_a"] for rec in records for rt in rec["retokenizations"] if rt["p"] == p]
             for p in p_values]
data_by_p.append(all_slopes_b)
labels = [f"p={p}" for p in p_values] + ["no prefix"]
positions = list(range(len(data_by_p)))
parts = ax.violinplot(data_by_p, positions=positions, widths=0.75,
                      showmeans=False, showmedians=False, showextrema=False)
for body in parts["bodies"]:
    body.set_facecolor("#cccccc"); body.set_edgecolor("none"); body.set_alpha(0.55)
ax.boxplot(data_by_p, positions=positions, widths=0.18, patch_artist=True,
           showfliers=False, medianprops=dict(color="black", lw=1.4),
           boxprops=dict(facecolor="white", edgecolor="black"),
           whiskerprops=dict(color="black"), capprops=dict(color="black"))
means = [np.mean(v) for v in data_by_p]
ax.plot(positions, means, "o-", color="crimson", lw=1.6, ms=7, zorder=5, label="mean")
ax.set_xticks(positions); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("entropy rate (nats/tok)"); ax.set_title("D) Entropy rate by condition")
ax.grid(True, axis="y", alpha=0.3); ax.legend()

fig.suptitle(f"OLMo-2-7B \u2014 entropy rate with/without retokenized prefix\n"
             f"({len(records)} passages \u00d7 {d['config']['num_retokenizations']} retokenizations)", y=1.01)
fig.tight_layout()
plt.show()

## 4. All conditions — controls, random init, training sweep

In [ ]:
CHECKPOINTS = [
    ("random_init", 0, "random"),
    ("ckpt_step1000", 1000, "5B"),
    ("ckpt_step5000", 5000, "21B"),
    ("ckpt_step10000", 10000, "42B"),
    ("ckpt_step25000", 25000, "105B"),
    ("ckpt_step50000", 50000, "210B"),
    ("ckpt_step101000", 101000, "424B"),
    ("ckpt_step200000", 200000, "839B"),
    ("ckpt_step400000", 400000, "1678B"),
    ("ckpt_step600000", 600000, "2517B"),
    ("ckpt_step800000", 800000, "3356B"),
    ("ckpt_step928000", 928000, "3893B"),
    ("controls_base", 935000, "final"),
    ("posttrain_sft", 940000, "SFT"),
    ("posttrain_dpo", 945000, "DPO"),
    ("posttrain_instruct", 950000, "Instruct"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

ax = axes[0, 0]
controls = load_json(WIKI, "controls_base")
if controls:
    conditions = ["retok", "random_text", "shuffled"]
    cond_labels = ["retok\n(same text)", "random text\n(different passage)", "shuffled\n(scrambled order)"]
    data_list = [get_slopes(controls, c)[0] for c in conditions]
    data_list.append([rec["slope_b"] for rec in controls["records"]])
    cond_labels.append("no prefix")
    positions = list(range(len(data_list)))
    parts = ax.violinplot(data_list, positions=positions, widths=0.75, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]: body.set_facecolor("#cccccc"); body.set_edgecolor("none"); body.set_alpha(0.55)
    ax.boxplot(data_list, positions=positions, widths=0.18, patch_artist=True, showfliers=False,
               medianprops=dict(color="black", lw=1.4), boxprops=dict(facecolor="white", edgecolor="black"),
               whiskerprops=dict(color="black"), capprops=dict(color="black"))
    means = [np.mean(v) for v in data_list]
    ax.plot(positions, means, "o-", color="crimson", lw=1.6, ms=7, zorder=5, label="mean")
    ax.set_xticks(positions); ax.set_xticklabels(cond_labels, fontsize=8)
    ax.set_ylabel("entropy rate (nats/tok)"); ax.set_title("Controls: base model, p=1.0")
    ax.grid(True, axis="y", alpha=0.3); ax.legend()

ax = axes[0, 1]
rand_data = load_json(WIKI, "random_init")
base_data = load_json(WIKI, "controls_base")
if rand_data and base_data:
    sa_r, sb_r = get_slopes(rand_data, "retok")
    sa_b, sb_b = get_slopes(base_data, "retok")
    data_list = [sa_b, sa_r, sb_b, sb_r]
    labels = ["base\nw/ prefix", "random init\nw/ prefix", "base\nno prefix", "random init\nno prefix"]
    positions = list(range(4))
    parts = ax.violinplot(data_list, positions=positions, widths=0.75, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]: body.set_facecolor("#cccccc"); body.set_edgecolor("none"); body.set_alpha(0.55)
    ax.boxplot(data_list, positions=positions, widths=0.18, patch_artist=True, showfliers=False,
               medianprops=dict(color="black", lw=1.4), boxprops=dict(facecolor="white", edgecolor="black"),
               whiskerprops=dict(color="black"), capprops=dict(color="black"))
    means = [np.mean(v) for v in data_list]
    ax.plot(positions, means, "o-", color="crimson", lw=1.6, ms=7, zorder=5, label="mean")
    ax.set_xticks(positions); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("entropy rate (nats/tok)"); ax.set_title("Random init vs pretrained base")
    ax.grid(True, axis="y", alpha=0.3); ax.legend()

ax3, ax4 = axes[1, 0], axes[1, 1]
steps_plot, mean_sa, mean_sb, std_sa, tok_labels = [], [], [], [], []
for name, step, tokens in CHECKPOINTS:
    data = load_json(WIKI, name)
    if data is None: continue
    sa, sb = get_slopes(data, "retok")
    if not sa: continue
    steps_plot.append(max(step, 1)); mean_sa.append(np.mean(sa)); mean_sb.append(np.mean(sb))
    std_sa.append(np.std(sa)); tok_labels.append(tokens)

if steps_plot:
    ax3.plot(steps_plot, mean_sa, "o-", color="crimson", lw=2, ms=7, label="with retok prefix")
    ax3.fill_between(steps_plot, [m-s for m,s in zip(mean_sa,std_sa)], [m+s for m,s in zip(mean_sa,std_sa)], alpha=0.15, color="crimson")
    ax3.plot(steps_plot, mean_sb, "s--", color="steelblue", lw=1.5, ms=5, label="no prefix")
    for s,m,t in zip(steps_plot,mean_sa,tok_labels): ax3.annotate(t,(s,m),textcoords="offset points",xytext=(0,10),fontsize=6,ha="center",color="gray")
    ax3.set_xscale("log"); ax3.set_xlabel("training step"); ax3.set_ylabel("mean entropy rate (nats/tok)")
    ax3.set_title("Entropy rate vs training step"); ax3.grid(True, alpha=0.3); ax3.legend()
    frac_gains = [1 - a/b for a,b in zip(mean_sa, mean_sb)]
    ax4.plot(steps_plot, frac_gains, "o-", color="darkgreen", lw=2, ms=7)
    for s,f,t in zip(steps_plot,frac_gains,tok_labels): ax4.annotate(t,(s,f),textcoords="offset points",xytext=(0,8),fontsize=6,ha="center",color="gray")
    ax4.set_xscale("log"); ax4.set_xlabel("training step"); ax4.set_ylabel("\u0394H / H(E0)  (fraction resolved)")
    ax4.set_title("Tokenization invariance vs training step")
    ax4.set_ylim(-0.1, 1.05); ax4.axhline(1.0, color="black", ls=":", lw=0.8, alpha=0.5)
    ax4.axhline(0.0, color="black", ls=":", lw=0.8, alpha=0.5); ax4.grid(True, alpha=0.3)

fig.suptitle("OLMo-2-7B \u2014 information gain from retokenized prefix\n(50 passages \u00d7 20 retokenizations, p=1.0)", y=1.01)
fig.tight_layout()
plt.show()

## 5. Forward vs reverse information gain (wikitext)

In [ ]:
fwd = load_json(WIKI, "controls_base")
rev = load_json(WIKI, "reverse_base")

if fwd and rev:
    fwd_sa = [rt["slope_a"] for rec in fwd["records"] for rt in rec["conditions"]["retok"] if math.isfinite(rt["slope_a"])]
    fwd_sb = [rec["slope_b"] for rec in fwd["records"]]
    rev_sa = [rt["slope_a_per_retok_tok"] for rec in rev["records"] for rt in rec["retokenizations"] if math.isfinite(rt["slope_a_per_retok_tok"])]
    rev_sb = [rt["slope_b_per_retok_tok"] for rec in rev["records"] for rt in rec["retokenizations"] if math.isfinite(rt["slope_b_per_retok_tok"])]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
    for ax, data, fc, labels, title in [
        (axes[0], [fwd_sa, fwd_sb], "#aec7e8", ["w/ retok\nprefix", "no prefix"], "Forward: predict E0 from E'"),
        (axes[1], [rev_sa, rev_sb], "#ffbb78", ["w/ canon\nprefix", "no prefix"], "Reverse: predict E' from E0")]:
        parts = ax.violinplot(data, positions=[0,1], widths=0.7, showmeans=False, showmedians=False, showextrema=False)
        for body in parts["bodies"]: body.set_facecolor(fc); body.set_edgecolor("none"); body.set_alpha(0.6)
        ax.boxplot(data, positions=[0,1], widths=0.18, patch_artist=True, showfliers=False,
                   medianprops=dict(color="black",lw=1.4), boxprops=dict(facecolor="white",edgecolor="black"),
                   whiskerprops=dict(color="black"), capprops=dict(color="black"))
        ms = [np.mean(v) for v in data]
        ax.plot([0,1], ms, "o-", color="crimson", lw=1.6, ms=8, zorder=5)
        for i,m in enumerate(ms): ax.text(i+0.15, m, f"{m:.2f}", fontsize=9, color="crimson", va="center")
        ax.set_xticks([0,1]); ax.set_xticklabels(labels, fontsize=10)
        ax.set_title(title); ax.set_ylim(bottom=-0.2); ax.grid(True, axis="y", alpha=0.3)
    axes[0].set_ylabel("nats / canon token"); axes[1].set_ylabel("nats / retok token")

    ax = axes[2]
    fg = 1 - np.mean(fwd_sa)/np.mean(fwd_sb); rg = 1 - np.mean(rev_sa)/np.mean(rev_sb)
    bars = ax.bar([0,1], [fg,rg], width=0.5, color=["#aec7e8","#ffbb78"], edgecolor="black", lw=0.8)
    ax.set_xticks([0,1]); ax.set_xticklabels(["Forward\n(predict E0|E')","Reverse\n(predict E'|E0)"], fontsize=10)
    ax.set_ylabel("\u0394H / H  (fraction resolved)"); ax.set_title("Fractional information gain")
    ax.set_ylim(0, 1.05); ax.axhline(1.0, color="black", ls=":", lw=0.8, alpha=0.5)
    for bar,val in zip(bars,[fg,rg]): ax.text(bar.get_x()+bar.get_width()/2, val+0.03, f"{val:.1%}", ha="center", fontsize=11, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle("OLMo-2-7B \u2014 forward vs reverse information gain (wikitext, p=1.0)", y=1.02)
    fig.tight_layout(); plt.show()
    print(f"Forward gain: {fg:.3f}  Reverse gain: {rg:.3f}")

## 6. Canonical vs retokenized prefix — entropy decomposition (wikitext)

In [ ]:
cp = load_json(WIKI, "canon_prefix")

if cp:
    slopes_no = [rec["slope_no_prefix"] for rec in cp["records"]]
    slopes_canon = [rec["slope_canon_prefix"] for rec in cp["records"]]
    slopes_retok = [rec["slope_retok_prefix_mean"] for rec in cp["records"]]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ax = axes[0]
    data = [slopes_canon, slopes_retok, slopes_no]
    labels = ["canonical\nprefix (E0)", "retokenized\nprefix (E')", "no prefix"]
    positions = [0, 1, 2]
    parts = ax.violinplot(data, positions=positions, widths=0.7, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]: body.set_facecolor("#cccccc"); body.set_edgecolor("none"); body.set_alpha(0.55)
    ax.boxplot(data, positions=positions, widths=0.18, patch_artist=True, showfliers=False,
               medianprops=dict(color="black",lw=1.4), boxprops=dict(facecolor="white",edgecolor="black"),
               whiskerprops=dict(color="black"), capprops=dict(color="black"))
    rng = random.Random(0)
    for i, vals in enumerate(data):
        xs = [positions[i] + (rng.random()-0.5)*0.18 for _ in vals]
        ax.scatter(xs, vals, s=12, alpha=0.4, color="#1f77b4", edgecolor="none", zorder=1)
    means = [np.mean(v) for v in data]
    ax.plot(positions, means, "o-", color="crimson", lw=1.6, ms=8, zorder=5, label="mean")
    for i,m in enumerate(means): ax.text(positions[i]+0.2, m, f"{m:.3f}", fontsize=9, color="crimson", va="center")
    ax.set_xticks(positions); ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel("entropy rate (nats / canon token)"); ax.set_title("Predicting E0: effect of prefix type")
    ax.grid(True, axis="y", alpha=0.3); ax.legend()

    ax = axes[1]
    mn, mc, mr = np.mean(slopes_no), np.mean(slopes_canon), np.mean(slopes_retok)
    content = mn - mr; tok_unc = mr - mc; residual = mc
    ax.bar(0, content, 0.5, color="#2ca02c", alpha=0.7, label=f"content resolved: {content:.3f}")
    ax.bar(0, tok_unc, 0.5, bottom=content, color="#ff7f0e", alpha=0.7, label=f"tokenization uncertainty: {tok_unc:.3f}")
    ax.bar(0, residual, 0.5, bottom=content+tok_unc, color="#d62728", alpha=0.7, label=f"residual: {residual:.3f}")
    ax.axhline(mn, color="black", ls="--", lw=1, alpha=0.5)
    ax.text(0.4, mn+0.03, f"no prefix: {mn:.3f}", fontsize=8)
    ax.set_xticks([0]); ax.set_xticklabels(["H(E0 | E')"], fontsize=10)
    ax.set_ylabel("nats / canon token"); ax.set_title("Entropy decomposition")
    ax.legend(loc="upper right", fontsize=8); ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle("OLMo-2-7B \u2014 canonical vs retokenized prefix (wikitext)\nIsolating tokenization uncertainty (50 passages, p=1.0)", y=1.02)
    fig.tight_layout(); plt.show()
    print(f"Content: {content/mn:.1%}  Tok uncertainty: {tok_unc/mn:.1%}  Residual: {residual/mn:.1%}")

## 7. Full prefix × target entropy matrix (wikitext)

In [ ]:
cp = load_json(WIKI, "canon_prefix")
rev = load_json(WIKI, "reverse_base")
rr = load_json(WIKI, "retok_retok")

if cp and rev and rr:
    e0_cold = np.mean([rec["slope_no_prefix"] for rec in cp["records"]])
    e0_given_e0 = np.mean([rec["slope_canon_prefix"] for rec in cp["records"]])
    e0_given_ep = np.mean([rec["slope_retok_prefix_mean"] for rec in cp["records"]])
    ep_cold = np.mean([r["slope_b_per_retok2_tok"] for rec in rr["records"] for r in rec["diff_pairs"] if math.isfinite(r["slope_b_per_retok2_tok"])])
    ep_given_e0 = np.mean([r["slope_a_per_retok_tok"] for rec in rev["records"] for r in rec["retokenizations"] if math.isfinite(r["slope_a_per_retok_tok"])])
    ep_given_ep_same = np.mean([r["slope_per_retok_tok"] for rec in rr["records"] for r in rec["same_pairs"] if math.isfinite(r["slope_per_retok_tok"])])
    ep_given_ep_diff = np.mean([r["slope_a_per_retok2_tok"] for rec in rr["records"] for r in rec["diff_pairs"] if math.isfinite(r["slope_a_per_retok2_tok"])])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.3, 1]})
    ax = axes[0]; ax.axis("off")
    row_labels = ["E0 prefix", "E' prefix", "no prefix"]
    col_labels = ["\u2192 E0\n(canonical)", "\u2192 E'same\n(same retok)", "\u2192 E'diff\n(diff retok)"]
    values = [[e0_given_e0, None, ep_given_e0], [e0_given_ep, ep_given_ep_same, ep_given_ep_diff], [e0_cold, None, ep_cold]]
    max_val = 8.0; cmap = plt.get_cmap("RdYlGn_r")
    tx, ty, cw, ch = 0.25, 0.15, 0.22, 0.22
    for j, label in enumerate(col_labels): ax.text(tx+(j+0.5)*cw, ty+3*ch+0.02, label, ha="center", va="bottom", fontsize=10, fontweight="bold")
    for i, label in enumerate(row_labels): ax.text(tx-0.02, ty+(2.5-i)*ch, label, ha="right", va="center", fontsize=10, fontweight="bold")
    for i in range(3):
        for j in range(3):
            x, y, val = tx+j*cw, ty+(2-i)*ch, values[i][j]
            if val is None:
                ax.add_patch(mpatches.FancyBboxPatch((x,y),cw,ch,boxstyle="round,pad=0.02",facecolor="#f0f0f0",edgecolor="gray",lw=0.5))
                ax.text(x+cw/2,y+ch/2,"\u2014",ha="center",va="center",fontsize=12,color="gray")
            else:
                ax.add_patch(mpatches.FancyBboxPatch((x,y),cw,ch,boxstyle="round,pad=0.02",facecolor=cmap(val/max_val),edgecolor="black",lw=0.8,alpha=0.85))
                ax.text(x+cw/2,y+ch/2,f"{val:.2f}",ha="center",va="center",fontsize=13,fontweight="bold",color="white" if val>3 else "black")
    ax.set_xlim(-0.05,1.05); ax.set_ylim(0,1.05)
    ax.set_title("Entropy rate matrix (nats/tok)\nprefix \u00d7 target", fontsize=12, pad=15)

    ax = axes[1]
    conds = [("E0\u2192E0",e0_given_e0,"#2ca02c"),("E'\u2192E0",e0_given_ep,"#98df8a"),("E'\u2192E'same",ep_given_ep_same,"#ff7f0e"),
             ("E0\u2192E'",ep_given_e0,"#ffbb78"),("E'\u2192E'diff",ep_given_ep_diff,"#d62728"),("E0 cold",e0_cold,"#aec7e8"),("E' cold",ep_cold,"#c5b0d5")]
    bars = ax.barh(range(len(conds)), [c[1] for c in conds], color=[c[2] for c in conds], edgecolor="black", lw=0.5)
    ax.set_yticks(range(len(conds))); ax.set_yticklabels([c[0] for c in conds], fontsize=9)
    ax.set_xlabel("entropy rate (nats/tok)"); ax.set_title("All conditions ranked")
    ax.grid(True, axis="x", alpha=0.3); ax.invert_yaxis()
    for bar,c in zip(bars,conds): ax.text(c[1]+0.1, bar.get_y()+bar.get_height()/2, f"{c[1]:.2f}", va="center", fontsize=9)
    fig.suptitle("OLMo-2-7B \u2014 complete prefix \u00d7 target entropy matrix (wikitext, p=1.0)", y=1.02)
    fig.tight_layout(); plt.show()

## 8. Three-condition cumulative information — same passage

No prefix, retokenized prefix, and canonical prefix on the same wikitext passage.

In [ ]:
d3 = load_json(WIKI, "example_3conditions")

if d3:
    T = d3["L_canon"]
    positions = np.arange(1, T + 1)
    cum_no = np.cumsum([-lp for lp in d3["logprobs_no_prefix"]])
    cum_retok = np.cumsum([-lp for lp in d3["logprobs_retok_prefix"]])
    cum_canon = np.cumsum([-lp for lp in d3["logprobs_canon_prefix"]])
    slope_no = cum_no[-1] / T
    slope_retok = cum_retok[-1] / T
    slope_canon = cum_canon[-1] / T

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(positions, cum_no, "-", color="steelblue", lw=1.5, label=f"no prefix (h = {slope_no:.3f})")
    ax.plot(positions, cum_retok, "-", color="crimson", lw=1.5, label=f"retokenized prefix (h = {slope_retok:.3f})")
    ax.plot(positions, cum_canon, "-", color="forestgreen", lw=1.5, label=f"canonical prefix (h = {slope_canon:.3f})")
    ax.set_xlabel("token position t", fontsize=11)
    ax.set_ylabel("cumulative \u2212log P (nats)", fontsize=11)
    ax.set_title(f"OLMo-2-7B \u2014 cumulative information content under three prefix conditions\n"
                 f"Same wikitext passage, {T} canonical tokens", fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()

    print(f"Content resolved:  {slope_no - slope_retok:.4f} ({(slope_no-slope_retok)/slope_no:.1%})")
    print(f"Tok uncertainty:   {slope_retok - slope_canon:.4f} ({(slope_retok-slope_canon)/slope_no:.1%})")
    print(f"Residual:          {slope_canon:.4f} ({slope_canon/slope_no:.1%})")
else:
    print("Run code/cumulative_logprob_canon_prefix_example.py to generate this data.")

---
# Part II — Python code (codeparrot-clean)

## 9. Controls — retok vs random text vs shuffled (code)

In [ ]:
controls = load_json(CODE, "controls_base")

if controls:
    conditions = ["retok", "random_text", "shuffled"]
    cond_labels = ["retok\n(same code)", "random text\n(different file)",
                   "shuffled\n(scrambled order)"]
    data_list = [get_slopes(controls, c)[0] for c in conditions]
    data_list.append([rec["slope_b"] for rec in controls["records"]])
    cond_labels.append("no prefix")

    fig, ax = plt.subplots(figsize=(8, 6))
    positions = list(range(len(data_list)))
    parts = ax.violinplot(data_list, positions=positions, widths=0.75, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]: body.set_facecolor("#cccccc"); body.set_edgecolor("none"); body.set_alpha(0.55)
    ax.boxplot(data_list, positions=positions, widths=0.18, patch_artist=True, showfliers=False,
               medianprops=dict(color="black",lw=1.4), boxprops=dict(facecolor="white",edgecolor="black"),
               whiskerprops=dict(color="black"), capprops=dict(color="black"))
    means = [np.mean(v) for v in data_list]
    ax.plot(positions, means, "o-", color="crimson", lw=1.6, ms=7, zorder=5, label="mean")
    for i,m in enumerate(means): ax.text(i+0.15, m, f"{m:.3f}", fontsize=8, color="crimson", va="center")
    ax.set_xticks(positions); ax.set_xticklabels(cond_labels, fontsize=9)
    ax.set_ylabel("entropy rate (nats/tok)")
    ax.set_title("Controls: base model on Python code, p=1.0")
    ax.grid(True, axis="y", alpha=0.3); ax.legend()
    fig.tight_layout(); plt.show()

    sa, sb = get_slopes(controls, "retok")
    gain = 1 - np.mean(sa)/np.mean(sb)
    print(f"Information gain (retok): {gain:.3f} ({gain:.1%})")

## 10. Forward vs reverse information gain (code)

In [ ]:
fwd = load_json(CODE, "controls_base")
rev = load_json(CODE, "reverse_base")

if fwd and rev:
    fwd_sa = [rt["slope_a"] for rec in fwd["records"] for rt in rec["conditions"]["retok"] if math.isfinite(rt["slope_a"])]
    fwd_sb = [rec["slope_b"] for rec in fwd["records"]]
    rev_sa = [rt["slope_a_per_retok_tok"] for rec in rev["records"] for rt in rec["retokenizations"] if math.isfinite(rt["slope_a_per_retok_tok"])]
    rev_sb = [rt["slope_b_per_retok_tok"] for rec in rev["records"] for rt in rec["retokenizations"] if math.isfinite(rt["slope_b_per_retok_tok"])]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
    for ax, data, fc, labels, title, ylabel in [
        (axes[0], [fwd_sa,fwd_sb], "#aec7e8", ["w/ retok\nprefix","no prefix"], "Forward: predict E0 from E'", "nats / canon token"),
        (axes[1], [rev_sa,rev_sb], "#ffbb78", ["w/ canon\nprefix","no prefix"], "Reverse: predict E' from E0", "nats / retok token")]:
        parts = ax.violinplot(data, positions=[0,1], widths=0.7, showmeans=False, showmedians=False, showextrema=False)
        for body in parts["bodies"]: body.set_facecolor(fc); body.set_edgecolor("none"); body.set_alpha(0.6)
        ax.boxplot(data, positions=[0,1], widths=0.18, patch_artist=True, showfliers=False,
                   medianprops=dict(color="black",lw=1.4), boxprops=dict(facecolor="white",edgecolor="black"),
                   whiskerprops=dict(color="black"), capprops=dict(color="black"))
        ms = [np.mean(v) for v in data]
        ax.plot([0,1], ms, "o-", color="crimson", lw=1.6, ms=8, zorder=5)
        for i,m in enumerate(ms): ax.text(i+0.15, m, f"{m:.2f}", fontsize=9, color="crimson", va="center")
        ax.set_xticks([0,1]); ax.set_xticklabels(labels, fontsize=10)
        ax.set_ylabel(ylabel); ax.set_title(title); ax.set_ylim(bottom=-0.2); ax.grid(True, axis="y", alpha=0.3)

    ax = axes[2]
    fg = 1 - np.mean(fwd_sa)/np.mean(fwd_sb); rg = 1 - np.mean(rev_sa)/np.mean(rev_sb)
    bars = ax.bar([0,1], [fg,rg], width=0.5, color=["#aec7e8","#ffbb78"], edgecolor="black", lw=0.8)
    ax.set_xticks([0,1]); ax.set_xticklabels(["Forward\n(predict E0|E')","Reverse\n(predict E'|E0)"], fontsize=10)
    ax.set_ylabel("\u0394H / H  (fraction resolved)"); ax.set_title("Fractional information gain")
    ax.set_ylim(0,1.05); ax.axhline(1.0, color="black", ls=":", lw=0.8, alpha=0.5)
    for bar,val in zip(bars,[fg,rg]): ax.text(bar.get_x()+bar.get_width()/2, val+0.03, f"{val:.1%}", ha="center", fontsize=11, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle("OLMo-2-7B \u2014 forward vs reverse on Python code (p=1.0)\n50 passages \u00d7 20 retokenizations", y=1.02)
    fig.tight_layout(); plt.show()
    print(f"Forward gain: {fg:.3f}  Reverse gain: {rg:.3f}")

## 11. Canonical vs retokenized prefix — entropy decomposition (code)

In [ ]:
cp = load_json(CODE, "canon_prefix")

if cp:
    slopes_no = [rec["slope_no_prefix"] for rec in cp["records"]]
    slopes_canon = [rec["slope_canon_prefix"] for rec in cp["records"]]
    slopes_retok = [rec["slope_retok_prefix_mean"] for rec in cp["records"]]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ax = axes[0]
    data = [slopes_canon, slopes_retok, slopes_no]
    labels = ["canonical\nprefix (E0)", "retokenized\nprefix (E')", "no prefix"]
    positions = [0, 1, 2]
    parts = ax.violinplot(data, positions=positions, widths=0.7, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]: body.set_facecolor("#cccccc"); body.set_edgecolor("none"); body.set_alpha(0.55)
    ax.boxplot(data, positions=positions, widths=0.18, patch_artist=True, showfliers=False,
               medianprops=dict(color="black",lw=1.4), boxprops=dict(facecolor="white",edgecolor="black"),
               whiskerprops=dict(color="black"), capprops=dict(color="black"))
    rng = random.Random(0)
    for i, vals in enumerate(data):
        xs = [positions[i] + (rng.random()-0.5)*0.18 for _ in vals]
        ax.scatter(xs, vals, s=12, alpha=0.4, color="#1f77b4", edgecolor="none", zorder=1)
    means = [np.mean(v) for v in data]
    ax.plot(positions, means, "o-", color="crimson", lw=1.6, ms=8, zorder=5, label="mean")
    for i,m in enumerate(means): ax.text(positions[i]+0.2, m, f"{m:.3f}", fontsize=9, color="crimson", va="center")
    ax.set_xticks(positions); ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel("entropy rate (nats / canon token)"); ax.set_title("Predicting E0: effect of prefix type (code)")
    ax.grid(True, axis="y", alpha=0.3); ax.legend()

    ax = axes[1]
    mn, mc, mr = np.mean(slopes_no), np.mean(slopes_canon), np.mean(slopes_retok)
    content = mn - mr; tok_unc = mr - mc; residual = mc
    ax.bar(0, content, 0.5, color="#2ca02c", alpha=0.7, label=f"content resolved: {content:.3f}")
    ax.bar(0, tok_unc, 0.5, bottom=content, color="#ff7f0e", alpha=0.7, label=f"tokenization uncertainty: {tok_unc:.3f}")
    ax.bar(0, residual, 0.5, bottom=content+tok_unc, color="#d62728", alpha=0.7, label=f"residual: {residual:.3f}")
    ax.axhline(mn, color="black", ls="--", lw=1, alpha=0.5)
    ax.text(0.4, mn+0.03, f"no prefix: {mn:.3f}", fontsize=8)
    ax.set_xticks([0]); ax.set_xticklabels(["H(E0 | E')"], fontsize=10)
    ax.set_ylabel("nats / canon token"); ax.set_title("Entropy decomposition (code)")
    ax.legend(loc="upper right", fontsize=8); ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle("OLMo-2-7B \u2014 canonical vs retokenized prefix on Python code\nIsolating tokenization uncertainty (50 passages, p=1.0)", y=1.02)
    fig.tight_layout(); plt.show()
    print(f"Content: {content/mn:.1%}  Tok uncertainty: {tok_unc/mn:.1%}  Residual: {residual/mn:.1%}")

## 12. Full prefix × target entropy matrix (code)

In [ ]:
cp = load_json(CODE, "canon_prefix")
rev = load_json(CODE, "reverse_base")
rr = load_json(CODE, "retok_retok")

if cp and rev and rr:
    e0_cold = np.mean([rec["slope_no_prefix"] for rec in cp["records"]])
    e0_given_e0 = np.mean([rec["slope_canon_prefix"] for rec in cp["records"]])
    e0_given_ep = np.mean([rec["slope_retok_prefix_mean"] for rec in cp["records"]])
    ep_cold = np.mean([r["slope_b_per_retok2_tok"] for rec in rr["records"] for r in rec["diff_pairs"] if math.isfinite(r["slope_b_per_retok2_tok"])])
    ep_given_e0 = np.mean([r["slope_a_per_retok_tok"] for rec in rev["records"] for r in rec["retokenizations"] if math.isfinite(r["slope_a_per_retok_tok"])])
    ep_given_ep_same = np.mean([r["slope_per_retok_tok"] for rec in rr["records"] for r in rec["same_pairs"] if math.isfinite(r["slope_per_retok_tok"])])
    ep_given_ep_diff = np.mean([r["slope_a_per_retok2_tok"] for rec in rr["records"] for r in rec["diff_pairs"] if math.isfinite(r["slope_a_per_retok2_tok"])])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.3, 1]})
    ax = axes[0]; ax.axis("off")
    row_labels = ["E0 prefix", "E' prefix", "no prefix"]
    col_labels = ["\u2192 E0\n(canonical)", "\u2192 E'same\n(same retok)", "\u2192 E'diff\n(diff retok)"]
    values = [[e0_given_e0, None, ep_given_e0], [e0_given_ep, ep_given_ep_same, ep_given_ep_diff], [e0_cold, None, ep_cold]]
    max_val = 8.0; cmap = plt.get_cmap("RdYlGn_r")
    tx, ty, cw, ch = 0.25, 0.15, 0.22, 0.22
    for j, label in enumerate(col_labels): ax.text(tx+(j+0.5)*cw, ty+3*ch+0.02, label, ha="center", va="bottom", fontsize=10, fontweight="bold")
    for i, label in enumerate(row_labels): ax.text(tx-0.02, ty+(2.5-i)*ch, label, ha="right", va="center", fontsize=10, fontweight="bold")
    for i in range(3):
        for j in range(3):
            x, y, val = tx+j*cw, ty+(2-i)*ch, values[i][j]
            if val is None:
                ax.add_patch(mpatches.FancyBboxPatch((x,y),cw,ch,boxstyle="round,pad=0.02",facecolor="#f0f0f0",edgecolor="gray",lw=0.5))
                ax.text(x+cw/2,y+ch/2,"\u2014",ha="center",va="center",fontsize=12,color="gray")
            else:
                ax.add_patch(mpatches.FancyBboxPatch((x,y),cw,ch,boxstyle="round,pad=0.02",facecolor=cmap(val/max_val),edgecolor="black",lw=0.8,alpha=0.85))
                ax.text(x+cw/2,y+ch/2,f"{val:.2f}",ha="center",va="center",fontsize=13,fontweight="bold",color="white" if val>3 else "black")
    ax.set_xlim(-0.05,1.05); ax.set_ylim(0,1.05)
    ax.set_title("Entropy rate matrix (nats/tok)\nprefix \u00d7 target", fontsize=12, pad=15)

    ax = axes[1]
    conds = [("E0\u2192E0",e0_given_e0,"#2ca02c"),("E'\u2192E0",e0_given_ep,"#98df8a"),("E'\u2192E'same",ep_given_ep_same,"#ff7f0e"),
             ("E0\u2192E'",ep_given_e0,"#ffbb78"),("E'\u2192E'diff",ep_given_ep_diff,"#d62728"),("E0 cold",e0_cold,"#aec7e8"),("E' cold",ep_cold,"#c5b0d5")]
    bars = ax.barh(range(len(conds)), [c[1] for c in conds], color=[c[2] for c in conds], edgecolor="black", lw=0.5)
    ax.set_yticks(range(len(conds))); ax.set_yticklabels([c[0] for c in conds], fontsize=9)
    ax.set_xlabel("entropy rate (nats/tok)"); ax.set_title("All conditions ranked")
    ax.grid(True, axis="x", alpha=0.3); ax.invert_yaxis()
    for bar,c in zip(bars,conds): ax.text(c[1]+0.1, bar.get_y()+bar.get_height()/2, f"{c[1]:.2f}", va="center", fontsize=9)
    fig.suptitle("OLMo-2-7B \u2014 complete prefix \u00d7 target entropy matrix (Python code, p=1.0)", y=1.02)
    fig.tight_layout(); plt.show()

---
# Part III — Cross-domain comparison

## 13. Entropy decomposition — code vs wikitext

In [ ]:
wiki_cp = load_json(WIKI, "canon_prefix")
code_cp = load_json(CODE, "canon_prefix")

if wiki_cp and code_cp:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    for ax, data, title in [(axes[0], wiki_cp, "Wikitext (natural language)"),
                             (axes[1], code_cp, "codeparrot-clean (Python code)")]:
        mn = np.mean([r["slope_no_prefix"] for r in data["records"]])
        mc = np.mean([r["slope_canon_prefix"] for r in data["records"]])
        mr = np.mean([r["slope_retok_prefix_mean"] for r in data["records"]])
        content = mn - mr; tok_unc = mr - mc; residual = mc

        ax.bar(0, content, 0.5, color="#2ca02c", alpha=0.7, label=f"content: {content:.3f} ({content/mn:.1%})")
        ax.bar(0, tok_unc, 0.5, bottom=content, color="#ff7f0e", alpha=0.7, label=f"tok. uncertainty: {tok_unc:.3f} ({tok_unc/mn:.1%})")
        ax.bar(0, residual, 0.5, bottom=content+tok_unc, color="#d62728", alpha=0.7, label=f"residual: {residual:.3f} ({residual/mn:.1%})")
        ax.axhline(mn, color="black", ls="--", lw=1, alpha=0.5)
        ax.text(0.4, mn+0.03, f"H(E0) = {mn:.3f}", fontsize=8)
        ax.set_xticks([0]); ax.set_xticklabels(["H(E0 | E')"], fontsize=10)
        ax.set_ylabel("nats / canon token"); ax.set_title(title)
        ax.legend(loc="upper right", fontsize=8); ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle("OLMo-2-7B \u2014 entropy decomposition: code vs natural language", y=1.02)
    fig.tight_layout(); plt.show()

## 14. Cumulative −logprob trajectories — code vs wikitext

Per-token logprobs for 5 passages from each domain. Data: `code/examples.json`.

In [ ]:
examples_path = CODE / "examples.json"

if examples_path.exists():
    d = json.loads(examples_path.read_text())

    fig, ax = plt.subplots(figsize=(10, 7))
    max_len = min(min(r["L_canon"] for r in d["wikitext"]), min(r["L_canon"] for r in d["code"]))

    for domain, data, c_no, c_with, label_no, label_with in [
        ("wikitext", d["wikitext"], "#4878CF", "#6ACC65",
         "Wikitext \u2014 no prefix", "Wikitext \u2014 retok prefix"),
        ("code", d["code"], "#D65F5F", "#EE854A",
         "Code \u2014 no prefix", "Code \u2014 retok prefix"),
    ]:
        for rec in data:
            neg_no = np.cumsum([-lp for lp in rec["logprobs_no_prefix"]])
            neg_with = np.cumsum([-lp for lp in rec["logprobs_with_prefix"]])
            t = np.arange(1, len(neg_no) + 1)
            ax.plot(t, neg_no, color=c_no, alpha=0.15, lw=0.8, ls="--")
            ax.plot(t, neg_with, color=c_with, alpha=0.15, lw=0.8)

        cum_no_all = [np.cumsum([-lp for lp in r["logprobs_no_prefix"]])[:max_len] for r in data]
        cum_with_all = [np.cumsum([-lp for lp in r["logprobs_with_prefix"]])[:max_len] for r in data]
        mean_no = np.mean(cum_no_all, axis=0)
        mean_with = np.mean(cum_with_all, axis=0)
        t = np.arange(1, max_len + 1)

        ax.plot(t, mean_no, color=c_no, ls="--", lw=2.5,
                label=f"{label_no}  (h = {mean_no[-1]/max_len:.3f})")
        ax.plot(t, mean_with, color=c_with, lw=2.5,
                label=f"{label_with}  (h = {mean_with[-1]/max_len:.3f})")

    ax.set_xlabel("token position t", fontsize=11)
    ax.set_ylabel("cumulative \u2212log P (nats)", fontsize=11)
    ax.set_title(
        "OLMo-2-7B \u2014 cumulative \u2212logprob: code vs natural language\n"
        "Dashed = no prefix, solid = with retok prefix (p=1.0). "
        "Thin lines = individual passages.", fontsize=11)
    ax.set_xlim(0, max_len); ax.set_ylim(0, None)
    ax.legend(fontsize=9, loc="upper left")
    ax.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()
else:
    print(f"Not found: {examples_path}")
    print("Run code/cumulative_logprob_examples.py to generate this data.")